# 05 · Mjolnir limits – per-user token quotas
`llm-token-limit` enforces tokens-per-minute **per subscription**. `avengers-initiative` users (thor, loki, hulk, black-widow) get a small budget; `tony-stark` (`infinity-gauntlet`) a large one.
Exceeding it returns **429** – only for that user.

In [ ]:
import sys; sys.path.insert(0, '.')
import time
from heimdal_helpers import *

import openai
def burst(user, n=40):   # 40 calls x ~330 tokens: far above hulk's 5000 tokens/min, well below tony-stark's 100000
    ok = throttled = 0
    left = None   # tokens still in the per-minute budget after the last successful call
    for _ in range(n):
        try:
            r = ask("Write a 150-word story about Mjolnir.", user=user, max_completion_tokens=300); ok += 1
            left = r['remaining_tokens']
        except openai.RateLimitError as e:
            throttled += 1
    return {'user': user, 'ok': ok, 'throttled(429)': throttled, 'budget_left_after_last_ok': left}
pd.DataFrame([burst('hulk'), burst('tony-stark')])

### What did the gateway see? (last 5 min)
Every request incl. the 429s is in `ApiManagementGatewayLogs` (a few minutes of lag). Throttled requests never reach the model, so they are **not** in `ApiManagementGatewayLlmLog`.

In [ ]:
KQL_THROTTLING = """
ApiManagementGatewayLogs
| where TimeGenerated > ago(5m)
| summarize Requests = count(),
            Allowed = countif(ResponseCode == 200),
            Throttled_429 = countif(ResponseCode == 429),
            Other = countif(ResponseCode !in (200, 429))
    by User = ApimSubscriptionId
| order by Requests desc
"""
# gateway logs lag a few minutes: poll until the burst above shows up
deadline = time.time() + 12 * 60
while True:
    df = kql(KQL_THROTTLING, hours=1)
    if len(df) or time.time() > deadline: break
    print("no rows yet - waiting 30s for ingestion..."); time.sleep(30)
df

### Allowed vs throttled per minute
The budget refills every minute, so bursts show up as bars of 429s.

In [ ]:
df = kql("""
ApiManagementGatewayLogs
| where TimeGenerated > ago(15m)
| summarize Allowed = countif(ResponseCode == 200), Throttled_429 = countif(ResponseCode == 429)
    by User = ApimSubscriptionId, Minute = bin(TimeGenerated, 1m)
| order by Minute asc
""", hours=1)
if len(df):
    df.pivot_table(index='Minute', columns='User', values=['Allowed', 'Throttled_429'], fill_value=0).plot.bar(figsize=(10, 4), title='Allowed vs throttled per minute')
df

### Throttled by the gateway or by the model?

In [ ]:
# A 429 that never reached the model (BackendResponseCode empty) was the gateway's token limit;
# a 429 with BackendResponseCode == 429 came from the Foundry deployment itself.
kql("""
ApiManagementGatewayLogs
| where TimeGenerated > ago(15m) and ResponseCode == 429
| summarize Count = count() by User = ApimSubscriptionId, BackendResponseCode
""", hours=1)